In [ ]:
from pathlib import Path

import pandas as pd


# ============================================================================
# 12. ICLE TOPIC-BIAS AUDIT
# ============================================================================
# Brooke & Hirst (2012) documented that ICLE's essay topics are unevenly
# distributed across L1 sub-corpora -- e.g. French essays skew toward
# literature/religion/politics, Japanese essays skew toward personal
# narratives about learning English or travel -- and showed this inflates
# NLI accuracy for every feature type they tested, including POS n-grams and
# function words, not just lexical content. 11_leakage_audit_country_names
# already found a related but distinct effect: essays that literally name
# their own country/language. This notebook checks the topic-level version
# of that confound directly, using the actual essay prompts in ICLEv3's
# metadata (the "Title" column), for BOTH the LLM and the SVM, and tests
# whether it explains accuracy INDEPENDENTLY of the self-disclosure leak
# already found, by restricting every accuracy check to the CLEAN subset
# from 11 first.
#
# NOTE on how this differs from Brooke & Hirst's design: their experiment
# was topic-segregated cross-validation WITHIN ICLE. This project trains
# only on TOEFL11 and never trains on ICLE at all -- ICLE is a zero-shot
# external test set for the LLM and a held-out test set for the SVM. So a
# model can't have "learned" ICLE's specific topic-L1 associations from
# ICLE training data the way an SVM cross-validated within ICLE could. What
# IS still possible is that essay topic/content correlates with L1 in ways
# a zero-shot LLM (broad web pretraining) or a lexical SVM can exploit
# without doing any real structural analysis -- which is what this checks.
#
# Saves icle_topic_flags.csv (doc_id, topic_exclusive_to_own_label) so
# 10_full_comparison_2x2.ipynb can build a "doubly-clean" subset (excludes
# both this and 11's leak) without re-deriving the topic logic.

BASE_DIR = Path("..").resolve()
METADATA_PATH = BASE_DIR / "data/raw/ICLEV3/metadata.csv"
LEAKAGE_FLAGGED_PATH = BASE_DIR / "data/processed/results/leakage_flagged_essays.csv"
FULLTEXT_LLM_PATH = BASE_DIR / "data/processed/results/llm_predictions_fulltext.csv"
RESULTS_DIR = BASE_DIR / "data/processed/results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_TOPIC_SUMMARY_PATH = RESULTS_DIR / "icle_topic_exclusivity_summary.csv"
OUTPUT_TOPIC_BY_LABEL_PATH = RESULTS_DIR / "icle_topic_exclusivity_by_label.csv"
OUTPUT_TOPIC_ACCURACY_PATH = RESULTS_DIR / "icle_topic_accuracy_comparison.csv"
OUTPUT_SVM_TOPIC_ACCURACY_PATH = RESULTS_DIR / "icle_topic_accuracy_comparison_svm.csv"
OUTPUT_TOPIC_FLAGS_PATH = RESULTS_DIR / "icle_topic_flags.csv"

ICLE_LANG_MAP = {
    "German": "DEU", "French": "FRA", "Italian": "ITA", "Japanese": "JPN",
    "Korean": "KOR", "Spanish": "SPA", "Turkish": "TUR", "Chinese": "ZHO",
}
EXCLUSIVITY_THRESHOLD = 0.8  # a title where >= 80% of its essays share one L1 counts as "topic-exclusive"

print("Loading ICLEv3 metadata...")
meta = pd.read_csv(METADATA_PATH)
meta = meta[meta["Native language"].isin(ICLE_LANG_MAP)].copy()
meta["label"] = meta["Native language"].map(ICLE_LANG_MAP)
meta["doc_id"] = "ICLEV3_" + meta["File name"].astype(str).str.strip()
meta["Title"] = meta["Title"].fillna("MISSING_TITLE").astype(str).str.strip()

print(f"{len(meta)} ICLEv3 essays across the 8 target languages have metadata.")

# --- How exclusive is each essay Title to a single L1? ---
title_stats = meta.groupby("Title")["label"].agg(
    n="size",
    top_label=lambda s: s.value_counts().idxmax(),
    top_count=lambda s: s.value_counts().max(),
).reset_index()
title_stats["top_share"] = title_stats["top_count"] / title_stats["n"]
title_stats.to_csv(OUTPUT_TOPIC_SUMMARY_PATH, index=False)

n_titles = len(title_stats)
n_exclusive_titles = int((title_stats["top_share"] >= EXCLUSIVITY_THRESHOLD).sum())
print(
    f"\n{n_exclusive_titles} / {n_titles} distinct essay titles have >= {EXCLUSIVITY_THRESHOLD:.0%} "
    "of their essays from a single L1."
)

meta = meta.merge(title_stats[["Title", "top_label", "top_share"]], on="Title", how="left")
meta["topic_exclusive_to_own_label"] = (meta["top_share"] >= EXCLUSIVITY_THRESHOLD) & (meta["top_label"] == meta["label"])

meta[["doc_id", "topic_exclusive_to_own_label"]].to_csv(OUTPUT_TOPIC_FLAGS_PATH, index=False)
print(f"Saved: {OUTPUT_TOPIC_FLAGS_PATH}")

by_label = (
    meta.groupby("label")["topic_exclusive_to_own_label"]
    .agg(n="size", n_exclusive="sum")
    .assign(pct_exclusive=lambda d: (d["n_exclusive"] / d["n"] * 100).round(1))
    .reset_index()
)
by_label.to_csv(OUTPUT_TOPIC_BY_LABEL_PATH, index=False)
print("\n=== Share of essays whose topic is (>= 80%) exclusive to their OWN label, by language ===")
print(by_label.to_string(index=False))
print(f"Saved: {OUTPUT_TOPIC_BY_LABEL_PATH}")

# --- Load the self-disclosure flag from 11 once, reused below for both LLM and SVM checks ---
self_ref_ids = set()
if LEAKAGE_FLAGGED_PATH.exists():
    flagged = pd.read_csv(LEAKAGE_FLAGGED_PATH)
    self_ref_ids = set(flagged.loc[flagged["self_referential_in_text"] == True, "doc_id"])
else:
    print(f"\nWARNING: {LEAKAGE_FLAGGED_PATH.name} not found -- run 11 first.")
    print("Proceeding without excluding self-disclosing essays, so the checks below conflate the two effects.")

# --- Does topic exclusivity predict LLM accuracy, INDEPENDENT of the self-disclosure leak from 11? ---
if FULLTEXT_LLM_PATH.exists():
    llm_df = pd.read_csv(FULLTEXT_LLM_PATH)
    llm_df = llm_df[llm_df["error"].fillna("").eq("") & llm_df["predicted_l1"].notna()].copy()
    llm_df = llm_df.merge(meta[["doc_id", "topic_exclusive_to_own_label"]], on="doc_id", how="left")
    llm_df["topic_exclusive_to_own_label"] = llm_df["topic_exclusive_to_own_label"].astype("boolean").fillna(False)
    llm_df["self_referential_in_text"] = llm_df["doc_id"].isin(self_ref_ids)

    # Restrict to ICLE rows BEFORE computing the topic-exclusivity split.
    # topic_exclusive_to_own_label can only be True for ICLE rows (TOEFL11 has no topic
    # metadata here and was fillna(False) above), so leaving TOEFL11 rows in play silently
    # pads the "False" bucket with TOEFL11 accuracy that has nothing to do with ICLE topic
    # bias -- this previously made the reported "non-exclusive" accuracy partly reflect
    # TOEFL11 rather than ICLE. Mirrors the SVM section below, which already restricts to
    # dataset == "ICLEV3" for exactly this reason.
    if "eval_partition" in llm_df.columns:
        icle_llm_df = llm_df[llm_df["eval_partition"] == "iclev3_external_test"].copy()
    else:
        print(
            f"WARNING: {FULLTEXT_LLM_PATH.name} has no eval_partition column -- cannot restrict to "
            "ICLE rows. The topic split below will include TOEFL11 rows in the 'False' bucket, "
            "diluting the comparison."
        )
        icle_llm_df = llm_df.copy()

    clean_llm_df = icle_llm_df[~icle_llm_df["self_referential_in_text"]].copy()
    print(
        f"\nRestricting to ICLE rows in the CLEAN subset from 11 ({len(clean_llm_df)} / {len(icle_llm_df)} "
        "ICLE rows) to isolate the topic effect from the already-documented self-disclosure leak, without "
        "diluting it with TOEFL11 rows that have no topic-exclusivity signal at all."
    )

    clean_llm_df["correct"] = clean_llm_df["predicted_l1"] == clean_llm_df["true_l1"]
    acc_split = (
        clean_llm_df.groupby("topic_exclusive_to_own_label")["correct"].agg(n="size", accuracy="mean").reset_index()
    )
    acc_split.insert(0, "partition", "iclev3_external_test")
    acc_split.to_csv(OUTPUT_TOPIC_ACCURACY_PATH, index=False)

    print("\n=== Full-text LLM accuracy on the CLEAN, ICLE-only subset: topic-exclusive vs. not ===")
    print(acc_split.to_string(index=False))
    print(f"Saved: {OUTPUT_TOPIC_ACCURACY_PATH}")
else:
    print(f"\nSkipping the LLM accuracy comparison -- {FULLTEXT_LLM_PATH.name} not found. Run 08 first.")

# --- Same check for the SVM (masked + fulltext), ICLE rows only -- TOEFL11 has no topic metadata here ---
SVM_SOURCES = [
    ("SVM_masked", RESULTS_DIR / "svm_predictions_full.csv", "svm_predicted_l1"),
    ("SVM_fulltext", RESULTS_DIR / "svm_predictions_fulltext.csv", "svm_predicted_l1"),
]

svm_topic_rows = []
for system_name, path, pred_col in SVM_SOURCES:
    if not path.exists():
        print(f"\nSkipping {system_name}: {path.name} not found (run 04 and/or 07 first).")
        continue

    svm_df = pd.read_csv(path)
    svm_df = svm_df[svm_df["dataset"] == "ICLEV3"].copy()
    svm_df = svm_df.merge(meta[["doc_id", "topic_exclusive_to_own_label"]], on="doc_id", how="left")
    svm_df["topic_exclusive_to_own_label"] = svm_df["topic_exclusive_to_own_label"].astype("boolean").fillna(False)
    svm_df["self_referential_in_text"] = svm_df["doc_id"].isin(self_ref_ids)
    svm_df = svm_df[~svm_df["self_referential_in_text"]].copy()
    svm_df["correct"] = svm_df[pred_col] == svm_df["true_l1"]

    split = svm_df.groupby("topic_exclusive_to_own_label")["correct"].agg(n="size", accuracy="mean").reset_index()
    split.insert(0, "system", system_name)
    svm_topic_rows.append(split)

if svm_topic_rows:
    svm_topic_split = pd.concat(svm_topic_rows, ignore_index=True)
    svm_topic_split.to_csv(OUTPUT_SVM_TOPIC_ACCURACY_PATH, index=False)
    print("\n=== SVM accuracy on ICLE, CLEAN subset: topic-exclusive vs. not ===")
    print(svm_topic_split.to_string(index=False))
    print(f"Saved: {OUTPUT_SVM_TOPIC_ACCURACY_PATH}")
    print(
        "\nCompare SVM_masked's gap against SVM_fulltext's gap here, the same way 11 compared them for the "
        "self-disclosure leak -- this tells you whether topic bias hits the LLM specifically harder than the SVM, "
        "or affects both similarly."
    )
else:
    print("\nNo SVM prediction files found -- run 04 and 07 first.")
